# HW2 실험 실행 (VS Code + Colab 커널)

오른쪽 위 **Select Kernel → Colab → New Colab Server → GPU → T4**를 고른 뒤 위에서부터 실행합니다.
코드는 2번 셀에 들어 있어서 파일을 따로 올릴 필요가 없습니다. 실행 결과는 이 노트북에 저장되므로 Claude가 바로 읽을 수 있습니다.

In [1]:
import torch, platform
assert torch.cuda.is_available(), 'GPU가 없습니다. Colab 서버를 T4 GPU로 다시 만드세요.'
print(torch.cuda.get_device_name(0), '|', platform.node())

Tesla T4 | fa1c0af927c8


In [2]:
# 실험 코드 풀기 (hw2_code 폴더의 압축본. 코드가 바뀌면 Claude가 이 셀을 다시 만듭니다)
import base64, io, os, zipfile
BLOB = '''
...
ABMAaQUAAP73AQAAAA==
'''
zipfile.ZipFile(io.BytesIO(base64.b64decode(BLOB.replace('\n', '')))).extractall('/content')
os.chdir('/content/hw2_code')
OUT = '/content/hw2_results'
os.makedirs(OUT, exist_ok=True)
print(sorted(os.listdir('.')))

['README.md', 'config.json', 'data', 'kspeech', 'requirements.txt', 'stimuli']


In [3]:
!pip -q install -r requirements.txt
!python -c "import torch, transformers, datasets; print(torch.__version__, transformers.__version__, datasets.__version__)"

2.14.1+cu130 5.22.1 4.8.5


## 1. 파일럿 (학습 분할 50문항씩, 1,320회, 약 1–2분)

평정으로 선정한 문구(Q02, Q04, Q03, Q01) 중 앞의 두 묶음으로 돌립니다. 끝나면 아래 분석 셀에서 통과 여부(`neutral_pass`, `extraction_pass`)가 모두 `true`인지 확인합니다.

In [4]:
# 이전 파일럿 결과를 지우고 새 코드로 다시 실행
!rm -f /content/hw2_results/*pilot*
!cd /content/hw2_code && python -m kspeech.run --phase pilot --out /content/hw2_results 2>&1 | grep -v Warning | tail -60

Loading weights: 100%|██████████| 338/338 [00:02<00:00, 117.93it/s]
[ynat/score] 500 evaluations (0 already done)
  32/500  18.4 eval/s  eta 0.4 min
[ynat/gen] 160 evaluations (0 already done)
  32/160  38.9 eval/s  eta 0.1 min
[nli/score] 500 evaluations (0 already done)
  32/500  58.4 eval/s  eta 0.1 min
[nli/gen] 160 evaluations (0 already done)
  32/160  50.3 eval/s  eta 0.0 min
{
 "model": "Qwen/Qwen2.5-1.5B-Instruct",
 "model_revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
 "device": "cuda",
 "dtype": "torch.float16",
 "torch": "2.14.1+cu130",
 "transformers": "5.22.1",
 "python": "3.13.16",
 "seed": 20261020,
 "phase": "pilot",
 "gpu": "Tesla T4",
 "stimuli_provisional": false,
 "started": "2026-11-22 11:47:39",
 "score_ynat": {
  "n": 500,
  "seconds": 10.8,
  "peak_mem_gb": 3.33
 },
 "gen_ynat": {
  "n": 160,
  "seconds": 3.6,
  "peak_mem_gb": 3.33
 },
 "score_nli": {
  "n": 500,
  "seconds": 11.0,
  "peak_mem_gb": 3.43
 },
 "gen_nli": {
  "n": 160,
  "seconds": 3.9,
  

In [5]:
!cd /content/hw2_code && python -m kspeech.analyze --phase pilot --out /content/hw2_results

{
 "ynat": {
  "neutral_acc": 0.6,
  "neutral_gate": 0.4,
  "neutral_pass": true,
  "extraction_rate": 1.0,
  "extraction_pass": true
 },
 "nli": {
  "neutral_acc": 0.58,
  "neutral_gate": 0.5,
  "neutral_pass": true,
  "extraction_rate": 1.0,
  "extraction_pass": true
 }
}
ynat: n=50 acc={'YP': 0.65, 'YR': 0.61, 'BP': 0.64, 'BR': 0.61} neutral={'Y': 0.58, 'B': 0.62} D=-0.0100 [-0.0800, +0.0500] p=0.8751 p_holm=0.8809
nli: n=50 acc={'YP': 0.54, 'YR': 0.5, 'BP': 0.56, 'BR': 0.5} neutral={'Y': 0.58, 'B': 0.58} D=+0.0200 [-0.0200, +0.0600] p=0.4405 p_holm=0.8809


## 2. 본실험 (검증 분할, 약 28,000회, 10–20분 예상)

파일럿 통과 여부가 모두 `true`일 때만 실행합니다. 중간에 끊기면 같은 셀을 다시 실행하면 이어서 진행합니다.

In [6]:
!cd /content/hw2_code && python -m kspeech.run --phase main --out /content/hw2_results 2>&1 | grep --line-buffered -v Warning

Loading weights: 100%|██████████| 338/338 [00:02<00:00, 124.09it/s]
[ynat/score] 12880 evaluations (0 already done)
  32/12880  36.0 eval/s  eta 6.0 min
  672/12880  54.2 eval/s  eta 3.8 min
  1312/12880  53.4 eval/s  eta 3.6 min
  1952/12880  52.0 eval/s  eta 3.5 min
  2592/12880  50.6 eval/s  eta 3.4 min
  3232/12880  49.3 eval/s  eta 3.3 min
  3872/12880  48.0 eval/s  eta 3.1 min
  4512/12880  47.1 eval/s  eta 3.0 min
  5152/12880  46.6 eval/s  eta 2.8 min
  5792/12880  46.3 eval/s  eta 2.6 min
  6432/12880  46.0 eval/s  eta 2.3 min
  7072/12880  45.7 eval/s  eta 2.1 min
  7712/12880  45.4 eval/s  eta 1.9 min
  8352/12880  45.2 eval/s  eta 1.7 min
  8992/12880  45.0 eval/s  eta 1.4 min
  9632/12880  44.8 eval/s  eta 1.2 min
  10272/12880  44.7 eval/s  eta 1.0 min
  10912/12880  44.5 eval/s  eta 0.7 min
  11552/12880  44.4 eval/s  eta 0.5 min
  12192/12880  44.3 eval/s  eta 0.3 min
  12832/12880  44.2 eval/s  eta 0.0 min
[ynat/gen] 2240 evaluations (0 already done)
  32/2240  40.9 ev

In [7]:
!cd /content/hw2_code && python -m kspeech.analyze --phase main --out /content/hw2_results

ynat: n=700 acc={'YP': 0.621, 'YR': 0.619, 'BP': 0.62, 'BR': 0.617} neutral={'Y': 0.623, 'B': 0.629} D=+0.0004 [-0.0079, +0.0082] p=0.9660 p_holm=0.9660
nli: n=600 acc={'YP': 0.512, 'YR': 0.49, 'BP': 0.511, 'BR': 0.485} neutral={'Y': 0.528, 'B': 0.532} D=+0.0046 [-0.0063, +0.0150] p=0.4081 p_holm=0.8161


## 3. 결과 내보내기

마지막 셀까지 실행한 뒤 노트북을 저장(Cmd+S)합니다.

In [8]:
# 결과 내보내기: 결과 폴더를 압축해 이 셀의 출력에 글자로 남깁니다 (Claude가 노트북에서 복원)
import base64, io, tarfile
b = io.BytesIO()
with tarfile.open(fileobj=b, mode='w:gz') as t:
    t.add('/content/hw2_results', arcname='hw2_results')
s = base64.b64encode(b.getvalue()).decode()
print('RESULTS_B64_BEGIN'); print(s); print('RESULTS_B64_END'); print(len(s), 'chars')

RESULTS_B64_BEGIN
...
RESULTS_B64_END
1283084 chars
